In [1]:
#Для визуального отображения прогресса выполнения кода
from tqdm import tqdm
#Для взаимодействия с ОС (здесь нужна для работы с директориями файлов)
import os
#Для работы с датафреймами
import pandas as pd
#Для вычислений
import numpy as np
#Для нормализации данных (приведения к единому диапазону)
from sklearn.preprocessing import MinMaxScaler
#Для работы с моделью
import tensorflow as tf
#Для модели бустинга
import xgboost as xgb
#Для модели ALS библиотеки surprise
from surprise import Dataset, Reader, BaselineOnly
#Для оптимизации использования памяти
from functools import reduce
#Для копирования данных
import copy
#Для загрузки моделей:
import pickle
# Garbage Collector для периодической очистки памяти
import gc                         
gc.enable()

2025-05-21 14:21:24.532283: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-21 14:21:24.532328: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-21 14:21:24.557688: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-21 14:21:24.622243: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-21 14:21:25.509024: W tensorflow/compiler/tf2

Хотя по заданию класс требуется один, логичнее по содержанию заданий создать 2 класса - по работе с моделями и по работе с данными. Третьим классом объединим всё в один объект.

Класс получился в принципе самодостаточным, его можно использовать "целиком", запустив метод fit_and_predict, который связывает все внутренние методы , так и в рамках каждого метода по отдельности. Для его успешной работы достаточно подать id пользователей (или их список) в метод fit_and_predict и получить массив с предсказаниями; уже обученные модели и обработанные данные будут загружены автоматически. При этом встроена возможность поменять достаточно многое - параметры предсказания, параметры обучения, собственно возможность переобучения, данные на которых всё обучается и т.д.

In [2]:
#Итоговый класс retrain model
#Сохраним здесь пути для предобученных моделей:
abs_path = os.path.dirname('.')
xgb_ranker_path = os.path.join(abs_path, "xgb_ranker.pkl")
als_model_path = os.path.join(abs_path, "als_model.pkl")
#Данные по транзакциям и продуктам:
transactions_csv_path = os.path.join(abs_path, './data/recsys/transactions.csv')
products_csv_path = os.path.join(abs_path, './data/recsys/products.csv')
#Для непосредственного получения предсказаний моделей нужны X и y с определенным набором столбцов:
X_train_path = os.path.join(abs_path, "./data/recsys/X_train_xgb.csv")
y_train_path = os.path.join(abs_path, "./data/recsys/y_train_xgb.csv")

class RecsysModel():

    
    def __init__(self, xgb_ranker = None, als_model = None, transactions = None, 
                 X_train = None,y_train = None, n = 10, r_xgb = 0.3, r_als =  0.7, user_treshold = 50):
        #Определим модели:
        self._xgb_ranker = xgb_ranker
        self._als_model = als_model
        
        #Также определим данные для подачи в модель (уже обработанные X и y):
        self._X_train = X_train
        self._y_train = y_train
        

        #Сет с транзакциями для нахождения наиболее популярных товаров (при необходимости):
        self._transactions = transactions
        
        #И значения по умолчанию для обучения моделей:
        self._n = n #к-во продуктов в предсказании, в условиях задания равно 10
        self._r_xgb =  r_xgb #xgb ratio - доля предсказания модели бустинга в итоговом предсказании, по умолчанию = 0.3
        self._r_als = r_als  #als ratio - доля предсказания модели ALS в итоговом предсказании, по умолчанию = 0.7
        self._user_treshold = user_treshold #минимальное к-во пользователей, заказавших товар, для предскзания по популярности
    
    
    #Методы для подгрузки данных:
    def load_xgb_model(self, new_model = None, new_model_path = None):
        #Если метод уже запускался, то ничего не вернем:
        if self._xgb_ranker is not None: 
            return self
        #Непосредственная загрузка из ноутбука:
        if new_model is not None:
            self._xgb_ranker = new_model
            return self
        #ЛИБО загрузка из файла в формате pkl:
        elif new_model_path is not None:
            with open(new_model_path, 'rb') as f:
                self._xgb_ranker = pickle.load(f)
            return self
        #По умолчанию загружается обученная предварительно модель:  
        if self._xgb_ranker is None: 
            with open(xgb_ranker_path, 'rb') as f:
                self._xgb_ranker = pickle.load(f)
     
    #Аналогично загрузим als модель:
    def load_als_model(self, new_model = None, new_model_path = None):
        if self._als_model is not None: 
            return self
        if new_model is not None:
            self._als_model = new_model
            return self
        elif new_model_path is not None:
            with open(new_model_path, 'rb') as f:
                    self._als_model = pickle.load(f)
            return self      
        if self._als_model is None:
            with open(als_model_path, 'rb') as f:
                self._als_model = pickle.load(f)
    
    #Для X и y оставим возможность загрузки непосредственно из ноутбука 
    #(так мне кажется логичнее из архитектуры этого и следующих классов):
    def load_X_train(self,  new_X = None):
        if self._X_train is not None: 
            return self
        if new_X is not None:
            self._X_train = new_X
            return self
        if self._X_train is None:
             self._X_train = pd.read_csv(X_train_path, sep = '\t')
    
    
    def load_y_train(self, new_y = None):
        if self._y_train is not None: 
            return self
        if new_y is not None:
            self._y_train = new_y
            return self
        if self._y_train is None:
             self._y_train = pd.read_csv(y_train_path, sep = '\t')
                
    # А для транзакций можно оставить 2 варианта с приоритетом подачи напрямую:            
    def load_transactions(self, new_set = None, new_path = None):
        if self._transactions is not None: 
            return self
        if new_set is not None:
            self._transactions = new_set
            return self
        elif new_path is not None:
            self._transactions = pd.read_csv(new_path)
            return self
        if self._transactions is None:    
            self._transactions = pd.read_csv(transactions_csv_path)
                
    #Теперь по порядку определимся с задачами
    #Предсказание по одной позиции:
    def predict_one(self, user_id):
        #Загрузка нужного:
        self.load_xgb_model()
        self.load_als_model()
        self.load_X_train()
        self.load_y_train()
        #Отделим части сета для подачи в модели:
        _tr_trim = self._X_train[self._X_train.user_id == user_id]
        _predset = self._y_train[self._y_train.user_id == user_id]
      
        #Получим итоговый массив для подачи в ALS:
        _reader = Reader(rating_scale=(1, 10)) # Зададим разброс оценок
        _als_test = Dataset.load_from_df(_predset, _reader)
        _als_test = _als_test.df.to_numpy().tolist()
        #Получаем предсказания по пользователю
        #Для XGBoost:
        _xgb_pred = self._xgb_ranker.predict(_tr_trim.set_index(['user_id', 'product_id']))
        #Для ALS:
        _predictions = self._als_model.test(_als_test)
        _als_pred = []
        for i in _predictions:
            _als_pred.append(i[3])
        _als_pred = np.array(_als_pred)
        #И сразу сохраним их в столбцы сета:
        _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
        _predset = _predset.merge(_predset
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
            .groupby('user_id')['predicted_rating'].nlargest(self._n)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
            .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
            how='right') # при этом все строки, которых нет в новом df удалятся
        
        #Получим наш список
        _model_preds = _predset.groupby('user_id')['product_id'].unique().reset_index().product_id.values[0]
        return _model_preds
    
    #На случай, если попадется новый пользователь, решим проблему холодного старта простейшей рекомендацией по популярности:
    def get_popularity_rec(self):
        self.load_transactions()
        #Сделаем сортировку по количеству перезаказов, при этом учитём также и тот факт, 
        #что товар должен быть покупаем немаленьким количеством пользователей (мы возьмем 50 пользователей минимум).
        _df = self._transactions[self._transactions.groupby('product_id')['user_id'].transform('nunique') > self._user_treshold]
        _popularity = _df.groupby('product_id')['reordered'].sum().reset_index()
        _popularity.sort_values('reordered', ascending=False, inplace=True)
        _res = np.array(_popularity.head(self._n).product_id, dtype=np.int32)
        return _res
 
    #Непосредственный метод для предсказаний:
    def predict(self, ids):
        self.load_X_train()
        #Проверка на тип входных значений для их обработки
        #Если на вход получен массив значений, то
        if hasattr(ids, "__len__"):
            #Зададим пустой массив, который мы будем подавать на выход
            _preds_array = np.zeros((len(ids), self._n), dtype=np.int32)
            for row, user_id in enumerate(ids):
                if user_id in self._X_train.user_id.unique().tolist():
                    _preds_array[row,:] = self.predict_one(user_id)
                else:
                    _preds_array[row,:] = np.array(self.get_popularity_rec())
            return _preds_array
        #Если был подан один пользователь
        else:
            if ids in self._X_train.user_id.unique().tolist():
                return self.predict_one(ids)
            else:
                return np.array(self.get_popularity_rec())
    
    
    #Обучение моделей на новых дынных:
    #Подадим в качестве значений по умолчанию параметры, которые дали в нашем случае наилучший результат:
    def refit_model(self, eval_metric = ['ndcg@10'],
                   lambdarank_num_pair_per_sample = 15, n_estimators =75, min_child_weight = 12, max_depth = 11,
                   learning_rate = 0.1, bsl_method = 'als', bsl_epochs = 20, reg_u = 18, reg_i = 6):
        self.load_X_train()
        self.load_y_train()
        X_xgb = self._X_train.set_index(['user_id', 'product_id'])
        y_xgb = self._y_train.set_index(['user_id', 'product_id'])
        #Определим XGBRanker:
        self._xgb_ranker = xgb.XGBRanker(objective ='rank:ndcg',
            lambdarank_pair_method ='topk',
            lambdarank_num_pair_per_sample = lambdarank_num_pair_per_sample,
            n_estimators = n_estimators,
            min_child_weight = min_child_weight,
            max_depth = max_depth,
            learning_rate = learning_rate,
            eval_metric= eval_metric,
            tree_method ='hist',
            device = 'cuda',
            random_state=42)
        
        #Обучение
        self._xgb_ranker.fit(
                X_xgb,
                y_xgb,
#                 eval_set= None,
                verbose = 0
            )
        
        print('XGBRanker has been successfully fit!')
        
        #Настроим данные для ALS:
        _train_als = self._y_train
        _train_als['rating'] =_train_als['rating'].astype(np.int32)
        
        _reader = Reader(rating_scale=(1, 10)) # Зададим разброс оценок

        _trainset = Dataset.load_from_df(_train_als, _reader)
        _trainset = _trainset.build_full_trainset()

        #Установим модель:
        _bsl_options =  {'method': bsl_method, 'n_epochs': bsl_epochs, 'reg_u': reg_u, 'reg_i': reg_i}
        self._als_model = BaselineOnly(bsl_options = _bsl_options)
        #Обучение
        self._als_model.fit(_trainset)
        print('ALS model has been successfully fit!')

    #Наконец, последний, объединяющий метод модели:
    #В модель подаём id пользователей для предсказания, а также, если необходимо переобучение модели меняем refit на True:
    def fit_and_predict(self, usr_ids, refit = False, eval_metric = ['ndcg@10'], lambdarank_num_pair_per_sample = 15, 
                        n_estimators =75, min_child_weight = 12, max_depth = 11, learning_rate = 0.1, bsl_method = 'als', 
                        bsl_epochs = 20, reg_u = 18, reg_i = 6):

        if refit:
            self.refit_model(eval_metric = eval_metric, lambdarank_num_pair_per_sample = lambdarank_num_pair_per_sample, 
                            n_estimators = n_estimators, min_child_weight = min_child_weight,
                            max_depth = max_depth, learning_rate = learning_rate, bsl_method = bsl_method, 
                            bsl_epochs = bsl_epochs, reg_u = reg_u, reg_i = reg_i)

        preds = self.predict(usr_ids)
        return preds

In [3]:
gc.collect()

43

In [4]:
#Check:
hybrid_model = RecsysModel()

In [5]:
hybrid_model.fit_and_predict([2,3,5,7])

/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [14:21:35] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_p

array([[34688, 24852, 48099, 44303, 13351, 16521,  4957, 13176, 47526,
         5212],
       [47766, 21137, 23650, 21903, 14992, 28373, 16797, 49683, 22035,
        40604],
       [24852, 13176, 21137, 21903, 47209, 47766, 27845, 27966, 47626,
        16797],
       [13176, 13802, 26346, 40852, 11520, 30391, 25199, 17638, 27690,
        47272]], dtype=int32)

In [6]:
#На одном экземпляре
hybrid_model.fit_and_predict(6) #6-го пользователя нет в сете, поэтому модель правильно возвращает n-популярных товаров

array([24852, 13176, 21137, 21903, 47209, 47766, 27845, 27966, 47626,
       16797], dtype=int32)

In [7]:
del hybrid_model
gc.collect()

28

Ниже написала класс для непосредственной обработки products и transactions, внесения в них изменений, объединения в общий сет, вычисления различных коэффициентов. Для работы можно как запускать все методы по отдельности, так и запустить объединяющий их метод preprocess_data, который на вход принимает изменения в транзакциях и продуктах (если такие есть), а выдает уже обработанные и готовые к подаче в модели X и y.

In [8]:
#Для работы с данными создадим отдельный класс:
class FormatDataset():
    def __init__(self,* , transactions = None, products = None, mainset = None, fill_empty = None, 
                 cols_to_leave = ['order_id', 'user_id', 'order_number','product_id', 'add_to_cart_order',
                                  'reordered',"order_dow","order_hour_of_day",'days_since_prior_order'], #что нужно для вычислений
                 feature_list = ['rating','pu_orders','p_reordered_times','pu_median_dspo',
                                 'pu_order_ratio','pu_median_cart_pos']):#что нужно для подачи в модель
        
        self._transactions = transactions
        self._products = products
        self._mainset = mainset
        self._fill_empty = fill_empty
        self._cols_to_leave = cols_to_leave
        self._feature_list = feature_list
        
    #Напишем пару классов для загрузки данных  
    # Проверка на ввод:  
    def process_input_data(self, new_data):
        #Dataframe возвращаем как есть:
        if isinstance(new_data, pd.DataFrame):
            return new_data
        #А путь передаем в pd.read_csv:
        elif isinstance(new_data, str):
            new_data = pd.read_csv(new_data)
            return new_data
        #Если что-то не так:
        else:
            raise TypeError('Wrong input type passed. Either pd.Dataframe or path to such is expected.')
            
    #Информация о продуктах:
    def load_products(self, new_set = None):
        if self._products is not None: 
            return self
        if new_set is not None:
            self._products = self.process_input_data(new_set)
        if self._products is None:
            self._products = pd.read_csv(products_csv_path)
            
    #Информация о транзакциях:
    def load_transactions(self, new_set = None):
        if self._transactions is not None: 
            return self
        if new_set is not None:
            self._products = self.process_input_data(new_set)
        if self._transactions is None:    
            self._transactions = pd.read_csv(transactions_csv_path)
            
    #И их объединенный сет:
    def load_mainset(self, new_set = None):
        if self._mainset is not None: 
            return self
        if new_set is not None:
            self._products = self.process_input_data(new_set)
    #Если mainset не определён, то запуститься метод merge_to_mainset:
        if self._mainset is None:
            self.load_products()
            self.load_transactions()
            self._mainset = self._merge_to_mainset()
            self._mainset = self._reduce_mem_usage(self._mainset)
    


    #Сюда вставим метод для уменьшения размерности датасета  c целью снижения нагрузки на память    
    def _reduce_mem_usage(self, df, verbose=True):
        numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
        start_mem = df.memory_usage().sum() / 1024**2    
        for col in df.columns:
            col_type = df[col].dtypes
            if col_type in numerics:
                c_min = df[col].min()
                c_max = df[col].max()
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        df[col] = df[col].astype(np.int64)  
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
                    else:
                        df[col] = df[col].astype(np.float64)    
        end_mem = df.memory_usage().sum() / 1024**2
        if verbose: 
            print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
        return df
 
    #Общий метод обработки изменений:
    def _renew_data(self, init_df, new_data):
        new_cols = None
    
        """
        new_data - датафрейм c новыми данными о транзакциях/продуктах
        """
        #Проверка на наличие исходных столбцов:
        diff1 = set(init_df.columns) - set(new_data.columns)
        if len(diff1)>0:
            raise Exception('New data missing ', list(diff1), 'column data')
        
        #Проверка на добавление новых столбцов:
        diff2 = set(new_data.columns) - set(init_df.columns)
    
        if len(diff2)>0:
            new_cols = list(diff2)
            print("New features: ", new_cols, ' column data to be added')  
    
        #Объединение данных:
        new_df = pd.concat([init_df,new_data], ignore_index=True)
    
        #Очистим память:
        del [init_df,new_data]
        gc.collect()
    
        #Заполним пустые колонки(если задано значение fill_empty):
        if (self._fill_empty is not None) & (new_cols is not None):
            new_df[new_cols] = new_df.loc[:,new_cols].fillna(value=self._fill_empty)
    
        if 'user_id' in new_df.columns:
            new_df = new_df.sort_values(by=['user_id', 'order_number'])
        else:
            new_df = new_df.sort_values(by=['product_id'])
        
        new_df = self._reduce_mem_usage(new_df)
        return new_df
    
    #Для конкретной ситуации
            
    #Изменение сета с транзакциями:
    def renew_transactions(self, new_data):
        self.load_transactions()
        new_data = self.process_input_data(new_data)
        self._transactions=self._renew_data(self._transactions, new_data)
        return self
    #Изменение данных о продуктах:    
    def renew_products(self, new_data):
        self.load_products()
        new_data = self.process_input_data(new_data)
        self._products=self._renew_data(self._products, new_data)
        return self
    
    #Объединение данных в mainset
    def _merge_to_mainset(self):
         # Собственно merge:
        df = reduce(lambda left, right: pd.merge(left, right, on='product_id', how='left'), [self._transactions,self._products])
        df = df[self._cols_to_leave] #удалим неиспользуемые колонки, если они есть в сете
        #Удаление позволяет снизить нагрузку на память :)
        
        #Заполним пустые значения в days_since_prior_order:
        df['days_since_prior_order']=df['days_since_prior_order'].fillna(
            df.groupby('user_id')['days_since_prior_order'].transform('mean'))
        #Здесь и далее вычислим все коэффициенты, которые мы использовали в моделях ранее по порядку
        #Абсолютно аналогично тому, что я делала для обучения модели:
        #Коэффициент номера заказа с учетом повышения актуальности каждого следущего по времени заказа:
        df['order_num_coef'] = df['order_number'].apply(lambda x: round((np.log10(x) + 1),2))
        #Перераспределим веса порядка заказа(перевернем порядок заказа, позиции с 11 и выше примут отрицательные значения).
        df['add_to_cart_coef'] = 11-df['add_to_cart_order']
        #Теперь скоректируем его с учетом номера заказа:
        df['add_to_cart_coef'] = df['add_to_cart_coef'] * df['order_num_coef']
        #Посчитаем коэффициент перезаказов c учетом add_to_cart_coef:
        df['reordered'] = df['reordered'] * df['add_to_cart_coef']
        # Посчитаем рейтинг, как add_to_cart_coef с учетом регулярности покупки продукта покупателем (reordered)
        # Доли показателей определим как 0.8 и 0.2 в результирующем соответственно.
        df['rating']  = (df.add_to_cart_coef * 0.8) + (df.reordered * 0.2)
        #Найдем сумму по всем позициям в связке продукт-пользователь:
        df['rating'] = df.groupby(['user_id','product_id'])['rating'].transform('sum')
        df['rating'] = df['rating'].clip(lower = 0) #удалим значения меньше нуля, как неактуальные
        #Воспользуемся MinMaxScaler для слишком больших значений рейтинга, причем сделаем это в группировке по пользователю:
        autoscaler = MinMaxScaler(feature_range = (1,10))
        def sc(row):
            return autoscaler.fit_transform(row.values.reshape(-1,1))

        df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
    
        #Вычислим ряд коэффициентов в группировке пользователь+продукт

        df['pu_orders'] = df.groupby(['user_id','product_id'])['order_number'].transform('count')
        df['max_orders'] = df.groupby(['user_id','product_id'])['order_number'].transform('max')
        df['min_orders'] = df.groupby(['user_id','product_id'])['order_number'].transform('min')
        df['pu_order_ratio'] = df['pu_orders']/(df['max_orders'] - df['min_orders'] + 1)
        df['pu_median_dspo'] = df.groupby(['user_id','product_id'])['days_since_prior_order'].transform('median')
        df['pu_median_cart_pos'] = df.groupby(['user_id','product_id'])['add_to_cart_order'].transform('median')
        df['p_reordered_times'] = df.groupby(['product_id'])['reordered'].transform('sum')
        num_cols = ['pu_orders','max_orders','min_orders','pu_order_ratio','pu_median_dspo','pu_median_cart_pos','p_reordered_times']
        df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce', downcast='float')
    
        #Округлим float, и удалим лишние колонки
        df = df.round(4)
        df.drop(columns = ['order_num_coef','max_orders','min_orders'], inplace = True)
        df = self._reduce_mem_usage(df)
        df = df.replace([np.inf, -np.inf], 0)
        #Очистка памяти:
        gc.collect()
        return df
           
    #Расчет признаков, относящихся к пользователю (user-based)
    def _get_feature_by_user(self, df):
        res = list()
        for i, v in tqdm(df.groupby('user_id')):
            res.append(
                (
                    i,
                    len(v['product_id']),
                    v['days_since_prior_order'].median(),
                    v['order_number'].max(),
                    v['add_to_cart_order'].max(),
                    v['order_hour_of_day'].median(),
                    v['order_dow'].median()
                )
            )
    
        res = pd.DataFrame(
            res,
            columns=[
                'user_id', 'u_prods_count', 'u_median_dspo' ,'u_max_orders', 'u_max_ordlen','u_median_hod', 'u_median_dow'
            ])
        #На всякий случай, заменим также inf (если такие будут) на нули:
        res = res.replace([np.inf, -np.inf], 0)

        res = self._reduce_mem_usage(res)
        return res
    
    #Расчет признаков, относящихся к продукту (product-based)
    def _get_feature_by_product(self,df):
        res = list()
        for i, v in tqdm(df.groupby('product_id')):
            res.append(
                (
                    i,
                    len(v['user_id']),
                    v['add_to_cart_order'].median(),
#                     v['reordered'].sum(),
                    v['order_dow'].median(),
                    v['order_hour_of_day'].median(),
                    v['days_since_prior_order'].median()
                )
            )
    
        res = pd.DataFrame(
            res,
            columns=[
                'product_id', 'p_user_cnt','p_median_cart_pos', 'p_median_dow', 'p_median_hod','p_median_dspo'])
        
        res = res.replace([np.inf, -np.inf], 0)

        res = self._reduce_mem_usage(res)
        return res
    
    
    #Расчет по всем показателям выше:
    def _get_model_features(self):
        self.load_mainset()
        print('Mainset loaded')
        df = self._mainset.groupby(['user_id', 'product_id'], as_index = False)\
            .agg({**{feats:'median' for feats in self._feature_list}})
        
        X_u = self._get_feature_by_user(self._mainset)
        print('User feats calculated')
        gc.collect()
        merged = reduce(lambda left, right: pd.merge(left, right, on='user_id', how='inner'), [X_u,df])
        
        print('User features merged')
    
        del [X_u,df]
        gc.collect()
    
        X_p = self._get_feature_by_product(self._mainset)
        print('Product feats calculated')
        gc.collect()
        merged = reduce(lambda left, right: pd.merge(left, right, on='product_id', how='inner'), [X_p,merged])
        print('Product features merged')
    
        del [X_p]
        gc.collect()
    
        merged.fillna(0, inplace=True)
    

        merged.sort_values(by=['user_id', 'product_id'], inplace=True)

        
        ordered_cols = ['user_id', 'product_id', 'p_user_cnt', 'p_median_cart_pos', 'p_reordered_times', 'p_median_dow', 
                        'p_median_hod', 'p_median_dspo', 'u_prods_count', 'u_median_dspo', 'u_max_orders', 'u_max_ordlen', 
                        'u_median_hod', 'u_median_dow', 'pu_orders', 'pu_median_dspo', 'pu_order_ratio', 'pu_median_cart_pos']
        
        features_cols = list(merged.drop(columns=['rating']).columns)
        
        #Для уже обученной модели важен порядок следования столбцов, поэтому:
        if set(ordered_cols) == set(features_cols):
            df_x = merged[ordered_cols]
        #А в случае если необходимо переобучить модель на новом сете:    
        else:
            df_x = merged[features_cols]

        df_x['qid'] = df_x['user_id']
    
        df_y = merged[['user_id', 'product_id','rating']]
        df_y['rating'] = df_y['rating'].apply(np.int64)
    
        del merged
        gc.collect()
        df_x = self._reduce_mem_usage(df_x)
        return df_x, df_y
            
    
    #И общий метод, который объединяет всё вышенаписанное:
    def preprocess_data(self, new_products = None, new_transactions = None):
        #Если меняем всё:
        if (new_products is not None) & (new_transactions is not None):
            self.renew_products(new_products).renew_transactions(new_transactions)
        #Если меняем одну часть:
        elif new_products is not None:
            self.renew_products(new_products)
        elif new_transactions is not None:
            self.renew_transactions(new_transactions)
            
        #Объединим products и transactions, вычислим необходимые столбцы и найдем X и y:
        X_train, y_train = self._get_model_features()
        return X_train, y_train

In [9]:
#Загрузим сеты, чтобы попробовать поизменять данные:
products = pd.read_csv(products_csv_path)
transactions = pd.read_csv(transactions_csv_path)

In [10]:
#Попробуем добавить новые колонки по транзакциям
new_tr = transactions.copy()
new_tr['new_col'] = np.nan

In [11]:
gc.collect()

0

In [12]:
#проверка
alter_transactions = FormatDataset(fill_empty = 0,transactions = transactions[:10000])
alter_transactions.preprocess_data(new_transactions = new_tr[:10000])[1].head(10)

New features:  ['new_col']  column data to be added
Mem. usage decreased to  0.53 Mb (68.2% reduction)
Mem. usage decreased to  0.84 Mb (26.7% reduction)


/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
/home/nette/miniconda3/lib/python3.11/site-packages/pandas/core/internals/blocks.py:1920: RuntimeWarning: overflow encountered in multiply
  values = self.values.round(decimals)  # type: ignore[union-attr]


Mem. usage decreased to  0.61 Mb (27.3% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████████████| 52/52 [00:00<00:00, 4122.24it/s]


Mem. usage decreased to  0.00 Mb (63.2% reduction)
User feats calculated
User features merged


100%|█████████████████████████████████████████████████████████████████████████████| 2638/2638 [00:00<00:00, 6317.55it/s]
/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to  0.04 Mb (61.0% reduction)
Product feats calculated
Product features merged
Mem. usage decreased to  0.17 Mb (0.0% reduction)


,user_id,product_id,rating
18,1,196,10
748,1,10258,7
756,1,10326,1
909,1,12427,8
949,1,13032,2
957,1,13176,1
1058,1,14084,1
1280,1,17122,1
1977,1,25133,6
2033,1,26088,1


In [13]:
#Попробуем добавить несколько продуктов:
pr_ids =  [49689, 49690, 49691, 49692]
pr_nms = ["Double Espresso Dark Roast Premium Blend", "Full English Breakfast", "French Cream Puff", 
          "Overpriced Turkish Chocolate, 1pcs"]
a_ids =  [26, 38, 8, 45]
d_ids =  [7, 1, 3, 19]
a_names = ["coffee", "frozen meals", "bakery desserts", "candy chocolate"]
d_names = ["beverages", "frozen", "bakery", "snacks"]

n_dict = {'product_id': pr_ids, 'product_name': pr_nms, 'aisle_id': a_ids, 'department_id': d_ids,
          'aisle': a_names,'department': d_names} 
   
pr_df = pd.DataFrame(n_dict)
pr_df

,product_id,product_name,aisle_id,department_id,aisle,department
0,49689,Double Espresso Dark Roast Premium Blend,26,7,coffee,beverages
1,49690,Full English Breakfast,38,1,frozen meals,frozen
2,49691,French Cream Puff,8,3,bakery desserts,bakery
3,49692,"Overpriced Turkish Chocolate, 1pcs",45,19,candy chocolate,snacks


In [14]:
alter_products = FormatDataset()
alter_products.preprocess_data(pr_df)[0].tail(10)

Mem. usage decreased to  1.47 Mb (35.4% reduction)


/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)


Mem. usage decreased to 931.83 Mb (67.0% reduction)
Mem. usage decreased to 931.83 Mb (0.0% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████| 100000/100000 [00:17<00:00, 5621.23it/s]


Mem. usage decreased to  1.43 Mb (59.5% reduction)
User feats calculated
User features merged


100%|███████████████████████████████████████████████████████████████████████████| 49465/49465 [00:12<00:00, 3930.34it/s]


Mem. usage decreased to  0.75 Mb (55.6% reduction)
Product feats calculated
Product features merged


/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to 496.15 Mb (0.0% reduction)


,user_id,product_id,p_user_cnt,p_median_cart_pos,p_reordered_times,p_median_dow,p_median_hod,p_median_dspo,u_prods_count,u_median_dspo,u_max_orders,u_max_ordlen,u_median_hod,u_median_dow,pu_orders,pu_median_dspo,pu_order_ratio,pu_median_cart_pos,qid
7575473,206209,40396,15080,6.0,78209.460938,3.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,2.0,25.140625,0.666504,4.5,206209
7593582,206209,40534,168,7.0,269.390015,3.0,14.0,8.0,129,20.28125,13,20.0,12.0,3.0,2.0,25.140625,0.666504,5.0,206209
7688747,206209,40992,3517,8.0,5929.040039,2.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,3.0,22.000000,0.600098,7.0,206209
7718770,206209,41213,2924,5.0,21032.560547,3.0,14.0,6.0,129,20.28125,13,20.0,12.0,3.0,7.0,29.000000,0.583496,3.0,206209
7807809,206209,41665,15547,8.0,42583.421875,2.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,1.0,30.000000,1.000000,5.0,206209
8263393,206209,43961,45911,8.0,135449.156250,2.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,3.0,22.000000,0.333252,10.0,206209
8330877,206209,44325,2559,8.0,2602.419922,3.0,13.0,8.0,129,20.28125,13,20.0,12.0,3.0,1.0,9.000000,1.000000,8.0,206209
9214364,206209,48370,3252,6.0,16332.759766,3.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,1.0,30.000000,1.000000,8.0,206209
9271090,206209,48697,7932,7.0,21991.099609,3.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,1.0,9.000000,1.000000,6.0,206209
9279693,206209,48742,1401,6.0,5400.930176,3.0,13.0,7.0,129,20.28125,13,20.0,12.0,3.0,2.0,13.500000,0.333252,9.0,206209


In [15]:
del [alter_transactions, alter_products]
gc.collect()

0

In [16]:
#И сразу всё поменяем:
alter_merge  = FormatDataset(fill_empty = 0,transactions = transactions[:10000]) 
alter_merge.preprocess_data(new_products = pr_df, new_transactions = new_tr[:10000])[0].head()

Mem. usage decreased to  1.47 Mb (35.4% reduction)
New features:  ['new_col']  column data to be added
Mem. usage decreased to  0.53 Mb (68.2% reduction)
Mem. usage decreased to  0.84 Mb (26.7% reduction)


/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
/home/nette/miniconda3/lib/python3.11/site-packages/pandas/core/internals/blocks.py:1920: RuntimeWarning: overflow encountered in multiply
  values = self.values.round(decimals)  # type: ignore[union-attr]


Mem. usage decreased to  0.61 Mb (27.3% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████████████| 52/52 [00:00<00:00, 5202.73it/s]


Mem. usage decreased to  0.00 Mb (63.2% reduction)
User feats calculated
User features merged


100%|█████████████████████████████████████████████████████████████████████████████| 2638/2638 [00:00<00:00, 6588.04it/s]
/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to  0.04 Mb (61.0% reduction)
Product feats calculated
Product features merged
Mem. usage decreased to  0.17 Mb (0.0% reduction)


,user_id,product_id,p_user_cnt,p_median_cart_pos,p_reordered_times,p_median_dow,p_median_hod,p_median_dspo,u_prods_count,u_median_dspo,u_max_orders,u_max_ordlen,u_median_hod,u_median_dow,pu_orders,pu_median_dspo,pu_order_ratio,pu_median_cart_pos,qid
18,1,196,134,1.0,1945.0000,2.0,11.0,8.0,118,20.265625,10,6.0,9.0,3.0,20.0,20.125,2.000000,1.0,1
748,1,10258,18,3.0,212.0000,3.0,9.0,20.0,118,20.265625,10,6.0,9.0,3.0,18.0,20.000,2.000000,3.0,1
756,1,10326,2,5.0,0.0000,4.0,15.0,28.0,118,20.265625,10,6.0,9.0,3.0,2.0,28.000,2.000000,5.0,1
909,1,12427,42,2.0,553.5000,2.0,11.0,14.0,118,20.265625,10,6.0,9.0,3.0,20.0,20.125,2.000000,2.5,1
949,1,13032,6,5.0,34.1875,3.0,8.0,20.0,118,20.265625,10,6.0,9.0,3.0,6.0,20.000,0.666504,6.0,1


In [17]:
del [alter_merge]
gc.collect()

0

In [18]:
#И просто ничего не меняя:
alter_none  = FormatDataset() 
alter_none.preprocess_data()[0].head(10)

/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)


Mem. usage decreased to 931.83 Mb (67.0% reduction)
Mem. usage decreased to 931.83 Mb (0.0% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████| 100000/100000 [00:18<00:00, 5535.99it/s]


Mem. usage decreased to  1.43 Mb (59.5% reduction)
User feats calculated
User features merged


100%|███████████████████████████████████████████████████████████████████████████| 49465/49465 [00:12<00:00, 3935.23it/s]


Mem. usage decreased to  0.75 Mb (55.6% reduction)
Product feats calculated
Product features merged


/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to 496.15 Mb (0.0% reduction)


,user_id,product_id,p_user_cnt,p_median_cart_pos,p_reordered_times,p_median_dow,p_median_hod,p_median_dspo,u_prods_count,u_median_dspo,u_max_orders,u_max_ordlen,u_median_hod,u_median_dow,pu_orders,pu_median_dspo,pu_order_ratio,pu_median_cart_pos,qid
33386,1,196,28819,2.0,3.975777e+05,3.0,12.0,7.394531,59,20.265625,10,9.0,9.0,3.0,10.0,20.125000,1.000000,1.0,1
1824032,1,10258,1656,3.0,2.003126e+04,2.0,11.0,7.000000,59,20.265625,10,9.0,9.0,3.0,9.0,20.000000,1.000000,3.0,1
1834644,1,10326,4324,3.0,4.955679e+04,2.0,12.0,8.000000,59,20.265625,10,9.0,9.0,3.0,1.0,28.000000,1.000000,5.0,1
2186672,1,12427,5515,3.0,6.495446e+04,2.0,11.0,7.000000,59,20.265625,10,9.0,9.0,3.0,10.0,20.125000,1.000000,2.5,1
2284420,1,13032,3080,4.0,2.533604e+04,3.0,12.0,7.000000,59,20.265625,10,9.0,9.0,3.0,3.0,20.000000,0.333252,6.0,1
2303253,1,13176,321553,3.0,3.815396e+06,2.0,13.0,7.000000,59,20.265625,10,9.0,9.0,3.0,2.0,21.500000,0.500000,6.0,1
2515563,1,14084,13911,4.0,1.440620e+05,3.0,13.0,7.000000,59,20.265625,10,9.0,9.0,3.0,1.0,20.265625,1.000000,2.0,1
3022241,1,17122,11188,4.0,9.506523e+04,2.0,12.0,7.000000,59,20.265625,10,9.0,9.0,3.0,1.0,28.000000,1.000000,6.0,1
4656078,1,25133,5515,5.0,4.057529e+04,2.0,12.0,7.000000,59,20.265625,10,9.0,9.0,3.0,8.0,20.500000,1.000000,4.0,1
4816919,1,26088,1983,5.0,1.150963e+04,3.0,13.0,8.000000,59,20.265625,10,9.0,9.0,3.0,2.0,17.625000,1.000000,4.5,1


In [19]:
del [alter_none]
gc.collect()

0

Теперь можно приступить к классу, котрый объединит два класса. Итоговый метод process_and_predict принимает на вход id пользователей для предсказания (обязательно для подачи), а также новые данные по транзакциям и продуктам (если есть) и выдает массив с предсказанием. При запуске самого класса можно также поменять ряд внутренних параметров, которые мы продублировали из классов выше - всё, что нужно для обучения модели (включая сами модели) и обработки данных.

In [20]:
#Итоговый класс retrain model
class MyRecsys():
    def __init__(self, xgb_ranker = None, als_model = None, transactions = None, products = None,
                  X_train = None,y_train = None, n = 10, r_xgb = 0.3, r_als =  0.7, user_treshold = 50, mainset = None,
                  fill_empty = None, cols_to_leave = ['order_id', 'user_id', 'order_number','product_id', 'add_to_cart_order',
                                  'reordered',"order_dow","order_hour_of_day",'days_since_prior_order'], #что нужно для вычислений
                  feature_list = ['rating','pu_orders','p_reordered_times','pu_median_dspo','pu_order_ratio','pu_median_cart_pos']):
            #Определим модели:
            self._xgb_ranker = xgb_ranker
            self._als_model = als_model
            #Определим данные:
            self._transactions = transactions
            self._products = products
            self._mainset = mainset
            #Также определим данные для подачи в модель (уже обработанные X и y):
            self._X_train = X_train
            self._y_train = y_train
            #И значения по умолчанию для обучения моделей:
            self._fill_empty = fill_empty
            self._cols_to_leave = cols_to_leave
            self._feature_list = feature_list
            
            self._n = n #к-во продуктов в предсказании, в условиях задания равно 10
            self._r_xgb =  r_xgb #xgb ratio - доля предсказания модели бустинга в итоговом предсказании, по умолчанию = 0.3
            self._r_als = r_als  #als ratio - доля предсказания модели ALS в итоговом предсказании, по умолчанию = 0.7
            self._user_treshold = user_treshold #минимальное к-во пользователей, заказавших товар, для предскзания по популярности
    
    #Запуск наших классов с определенным набором параметров
    #Для класса работы с моделями:
    def load_model_wrapper(self):
        model_wrapper = RecsysModel(xgb_ranker = self._xgb_ranker, als_model = self._als_model, 
                                    transactions = self._transactions, X_train = self._X_train, y_train = self._y_train, 
                                    n = self._n, r_xgb = self._r_xgb, r_als =  self._r_als, 
                                    user_treshold = self._user_treshold)
        return model_wrapper
            
    #Для класса работы с данными для моделей:
    def load_data_formatter(self):
        data_formatter = FormatDataset(transactions = self._transactions, products = self._products, mainset = self._mainset,
                                       fill_empty = self._fill_empty, cols_to_leave = self._cols_to_leave, 
                                       feature_list = self._feature_list)
        return data_formatter
            
    
    #И итоговый метод: 
    def process_and_predict(self, usr_ids, products_addition = None, transactions_addition = None, refit = False):
        #Загрузим класс для обработки дынных
        data_formatter = self.load_data_formatter()
        
        #Обновим X и y:
        self._X_train,  self._y_train = data_formatter.preprocess_data(new_products = products_addition, 
                                                                      new_transactions = transactions_addition)
        #Загрузка класса для работы с моделями:
        model_wrapper = self.load_model_wrapper()
        prediction = model_wrapper.fit_and_predict(usr_ids = usr_ids, refit = refit)
        return prediction

In [21]:
gc.collect()

0

In [22]:
# Проверка на общую работоспособность
# Подаем в модель урезанную версию транзакций (для скорости), определим замену Nan на нули, 
#а также уменьшим treshold для предсказания по популярности до 10 заказов пользователей (т.к. мы работаем с урезанным сетом):
recsys_wrap = MyRecsys(fill_empty = 0,transactions = transactions[:10000], user_treshold = 10)
recsys_wrap.process_and_predict(usr_ids = [2,3,5,7], products_addition = pr_df, transactions_addition = new_tr[:10000])

Mem. usage decreased to  1.47 Mb (35.4% reduction)
New features:  ['new_col']  column data to be added
Mem. usage decreased to  0.53 Mb (68.2% reduction)
Mem. usage decreased to  0.84 Mb (26.7% reduction)


/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
/home/nette/miniconda3/lib/python3.11/site-packages/pandas/core/internals/blocks.py:1920: RuntimeWarning: overflow encountered in multiply
  values = self.values.round(decimals)  # type: ignore[union-attr]


Mem. usage decreased to  0.61 Mb (27.3% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████████████| 52/52 [00:00<00:00, 4917.56it/s]


Mem. usage decreased to  0.00 Mb (63.2% reduction)
User feats calculated
User features merged


100%|█████████████████████████████████████████████████████████████████████████████| 2638/2638 [00:00<00:00, 6440.62it/s]
/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to  0.04 Mb (61.0% reduction)
Product feats calculated
Product features merged
Mem. usage decreased to  0.17 Mb (0.0% reduction)


/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

array([[24852, 47209, 18523, 33754, 32792, 13176, 47766, 16589, 22124,
        12000],
       [47766, 21903, 39190,  9387, 17668, 22035, 16797, 32402, 43961,
        23650],
       [13176, 24852, 47209, 21903, 24964, 21137, 22935,  8518, 26209,
        16797],
       [40852, 42803, 21137, 13198, 17638, 37602, 30391, 13176,  6361,
        43967]], dtype=int32)

In [23]:
gc.collect()

21

In [24]:
#Здесь попробуем переобучить модель на фиктивных новых данных:
recsys_wrap = MyRecsys(fill_empty = 0,transactions = transactions[:10000], user_treshold = 10)
recsys_wrap.process_and_predict(usr_ids = [2,3,5,7], transactions_addition = new_tr[:10000], refit = True)

New features:  ['new_col']  column data to be added
Mem. usage decreased to  0.53 Mb (68.2% reduction)
Mem. usage decreased to  0.84 Mb (26.7% reduction)


/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
/home/nette/miniconda3/lib/python3.11/site-packages/pandas/core/internals/blocks.py:1920: RuntimeWarning: overflow encountered in multiply
  values = self.values.round(decimals)  # type: ignore[union-attr]


Mem. usage decreased to  0.61 Mb (27.3% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████████████| 52/52 [00:00<00:00, 4189.31it/s]


Mem. usage decreased to  0.00 Mb (63.2% reduction)
User feats calculated
User features merged


100%|█████████████████████████████████████████████████████████████████████████████| 2638/2638 [00:00<00:00, 6681.87it/s]
/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to  0.04 Mb (61.0% reduction)
Product feats calculated
Product features merged
Mem. usage decreased to  0.17 Mb (0.0% reduction)
XGBRanker has been successfully fit!
Estimating biases using als...
ALS model has been successfully fit!


/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

array([[24852, 32792, 47209, 13176, 12000, 47766, 33754, 36735, 19156,
        18523],
       [39190, 47766, 21903,  9387, 17668, 22035,  1819, 18599, 16797,
        43961],
       [13176, 24852, 47209, 21903, 24964, 21137, 22935,  8518, 26209,
        16797],
       [37602, 21137, 39275, 13176, 17638, 39121, 40852,  4920, 30391,
         8277]], dtype=int32)

In [25]:
#Так, а теперь просто запустим без доп.параметров:
recsys_wrap = MyRecsys()
recsys_wrap.process_and_predict(usr_ids = [2,3,5,7])

/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)


Mem. usage decreased to 931.83 Mb (67.0% reduction)
Mem. usage decreased to 931.83 Mb (0.0% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████| 100000/100000 [00:18<00:00, 5533.50it/s]


Mem. usage decreased to  1.43 Mb (59.5% reduction)
User feats calculated
User features merged


100%|███████████████████████████████████████████████████████████████████████████| 49465/49465 [00:12<00:00, 3910.73it/s]


Mem. usage decreased to  0.75 Mb (55.6% reduction)
Product feats calculated
Product features merged


/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to 496.15 Mb (0.0% reduction)


/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

array([[24852, 32792, 47766, 47209, 13176, 12000,  1559, 19156, 16589,
        20574],
       [47766, 21903, 39190,  9387, 17668, 22035, 16797, 43961,  1819,
        28373],
       [24852, 13176, 21137, 21903, 47209, 47766, 27845, 27966, 47626,
        16797],
       [40852, 21137, 37602, 17638, 45628, 13176, 47272,  4920, 31683,
        39275]], dtype=int32)

Также попробуем добавить информацию напрямую из файла:

In [26]:
#Сохраним df с "новыми продуктами":
pr_df.to_csv('./data/recsys/pr_df.csv',index=False)

In [28]:
#И добавим их:
recsys_wrap = MyRecsys()
recsys_wrap.process_and_predict(usr_ids = [2,3,5,7], products_addition = './data/recsys/pr_df.csv')

Mem. usage decreased to  1.47 Mb (35.4% reduction)


/tmp/ipykernel_24592/1918688992.py:177: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  df['rating'] = df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)


Mem. usage decreased to 931.83 Mb (67.0% reduction)
Mem. usage decreased to 931.83 Mb (0.0% reduction)
Mainset loaded


100%|█████████████████████████████████████████████████████████████████████████| 100000/100000 [00:17<00:00, 5608.80it/s]


Mem. usage decreased to  1.43 Mb (59.5% reduction)
User feats calculated
User features merged


...

Mem. usage decreased to  0.75 Mb (55.6% reduction)
Product feats calculated
Product features merged


/tmp/ipykernel_24592/1918688992.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_y['rating'] = df_y['rating'].apply(np.int64)


Mem. usage decreased to 496.15 Mb (0.0% reduction)


/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _predset['predicted_rating'] = _xgb_pred * self._r_xgb + _als_pred * self._r_als
/tmp/ipykernel_24592/2258437369.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

array([[24852, 32792, 47766, 47209, 13176, 12000,  1559, 19156, 16589,
        20574],
       [47766, 21903, 39190,  9387, 17668, 22035, 16797, 43961,  1819,
        28373],
       [24852, 13176, 21137, 21903, 47209, 47766, 27845, 27966, 47626,
        16797],
       [40852, 21137, 37602, 17638, 45628, 13176, 47272,  4920, 31683,
        39275]], dtype=int32)

Отлично, всё работает. Осталось собрать из этого файл в формате .py и отправить всё на проверку